# JupyterHealth SoF Provider App
Device data for the launched patient, from JupyterHealth Exchange.

In [ ]:
# [SCAFFOLDED] launch context + data fetch — usually untouched
from provider_app import launch_context, jhe_auth, jhe_data, patient_resolver, identity

ctx = launch_context.current()
client = jhe_auth.client_for_launch(ctx)

# Two guardrails, both surfaced as a clean message (no traceback, no patient info):
#  - PatientNotInJHE: the launching provider's JHE account can't see this patient
#    (JupyterHealth RBAC) or the MRN isn't enrolled.
#  - identity.IdentityError: the resolved JHE record could not be confirmed to be
#    the same person as the launched EHR patient (jhe_data.fetch verifies this in
#    shared code, so it can't be bypassed here). Fails closed.
# access_note is None on success.
access_note = None
data = {}
try:
    data = jhe_data.fetch(ctx, client=client, types=["heart_rate", "sleep", "steps"])  # <-- edit to the data types you want
except patient_resolver.PatientNotInJHE:
    access_note = (
        f"You don't have access to this patient's data in JupyterHealth "
        f"(MRN {ctx.patient_mrn}). They may not be enrolled, or your JupyterHealth "
        f"account isn't authorized to view them."
    )
except identity.IdentityError:
    access_note = (
        "Patient identity could not be verified, or does not match the launched "
        "patient — refusing to display data."
    )

In [ ]:
# ──────── ADD YOUR ANALYTICS + VISUALIZATION BELOW ────────
# Example: plot the first requested data type. Replace this with your own analytics.
if access_note:
    print("🔒 " + access_note)
elif not data:
    print("No data returned for this patient.")
else:
    import plotly.express as px

    data_type, df = next(iter(data.items()))
    if df.empty:
        print(f"No {data_type} data for this patient.")
    else:
        value_cols = [c for c in df.columns if c.endswith("_value")]
        y = value_cols[0] if value_cols else df.columns[-1]
        fig = px.line(df, x="effective_time_frame_date_time", y=y, title=data_type)
        fig.show()